# 04. Type-Safe Structured Outputs with Pydantic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/04_structured_outputs.ipynb)

Prompting an LLM to "reply only in JSON" often breaks in production when responses include conversational preambles, trailing commas, or ` ```json ` Markdown code fences. Passing `response_mime_type="application/json"` together with `response_schema=<Pydantic BaseModel>` activates **grammar-constrained decoding**: Gemini restricts token sampling directly to the JSON Schema grammar derived from your model, and `response.parsed` returns an already-validated Pydantic instance with zero brittle regex or string stripping.

![Figure 4-1: Grammar-Constrained Structured Output Pipeline with Pydantic](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_structured_pipeline.png)

*Figure 4-1: Grammar-Constrained Structured Output Pipeline with Pydantic*

---

## 1. 3-Line Client Setup & Defining a Nested Pydantic Schema

First, initialize the `google-genai` client using your `GEMINI_API_KEY` stored in Google Colab Secrets (`🔑` panel on the left sidebar):

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

Next, compose a nested Pydantic schema (`ActionItem` inside `IncidentSummary`) to extract structured postmortem metadata from an unstructured operational incident log. Passing `IncidentSummary` to `response_schema` instructs the SDK to compile the Pydantic model into a strict schema and populate `response.parsed` automatically:

In [ ]:
from typing import Literal

from google.genai import types
from pydantic import BaseModel, Field


class ActionItem(BaseModel):
    owner: str = Field(description="Team or engineer responsible for the follow-up task.")
    task: str = Field(description="Concrete, actionable engineering remediation step.")
    priority: Literal["P0", "P1", "P2"] = Field(description="Urgency tier from P0 (immediate) to P2.")


class IncidentSummary(BaseModel):
    title: str = Field(description="Concise incident headline under 60 characters.")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(
        description="Overall customer-impact severity level."
    )
    root_cause: str = Field(description="Single-sentence technical root cause explanation.")
    affected_services: list[str] = Field(description="Microservice names impacted during the outage.")
    action_items: list[ActionItem] = Field(description="Prioritized follow-up engineering action items.")


raw_incident_log = """
[2026-04-12 14:03 UTC] PagerDuty Alert: 5xx error rate on checkout-api spiked to 38%.
[14:09 UTC] On-call SRE noticed PostgreSQL connection pool at 100% saturation.
[14:16 UTC] Root cause identified: a slow upstream payment-webhook timeout triggered an
unbounded client retry storm from payment-worker into checkout-api.
[14:24 UTC] Mitigated by enabling circuit-breaker shedding and restarting stalled workers.
Follow-ups agreed in retro:
1. SRE Team to cap exponential backoff and enforce circuit-breaker budgets immediately (P0).
2. Payments Team to add PgBouncer pool saturation alerts at 80% utilization (P1).
3. Platform Team to replay webhook timeout scenarios in staging load tests next sprint (P2).
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=raw_incident_log,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IncidentSummary,
        temperature=0.1,
    ),
)

# response.parsed is an already-validated IncidentSummary instance with full IDE autocomplete
summary: IncidentSummary = response.parsed
print(f"[{summary.severity}] {summary.title} ({len(summary.action_items)} action items)")
print(summary.model_dump_json(indent=2))

---

## 2. Schema Design Best Practices & Rendering Extracted Data with Drawlib

While grammar-constrained decoding guarantees syntactic validity, thoughtful Pydantic schema design dramatically improves **semantic accuracy** and **latency** on production workloads:

![Figure 4-2: Pydantic Schema Design Patterns for Reliable LLM Extraction](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_schema_best_practices.png)

*Figure 4-2: Pydantic Schema Design Patterns for Reliable LLM Extraction*

Keep these four engineering rules in mind when authoring schemas for `response_schema`:

1. **Use `Literal` or `Enum` for closed categories**: Constrain finite taxonomies (`Literal["P0", "P1", "P2"]`) so downstream routers and database `ENUM` columns never encounter unexpected casing or synonyms.
2. **Compose nested `list[SubModel]` instead of `dict[str, Any]`**: Arbitrary open dictionaries lack fixed property names and cannot be grammar-constrained reliably; model key-value pairs explicitly as `list[SubModel]`.
3. **Attach concise `Field(description=...)` metadata**: Descriptions are forwarded in the compiled JSON Schema, guiding Gemini on domain semantics (such as character budgets, ISO units, or severity thresholds) without bloating your main prompt.
4. **Tune `temperature` and `thinking_budget` for deterministic speed**: For straightforward entity extraction on `gemini-2.5-flash`, pairing `temperature=0.1` with `thinking_config=types.ThinkingConfig(thinking_budget=0)` minimizes tail latency and output variance.

Because `summary.action_items` is already a strongly typed `list[ActionItem]`, you can feed it straight into a **Drawlib `Table`** (`setup(..., cwd=True)` + `Table` + `show()`) to render an executive incident summary right inside your notebook cell:

In [ ]:
from drawlib.canvas import setup, show
from drawlib.smartarts import Table
from drawlib.styles import Colors, Styles
from drawlib.text import text

setup(width=120, height=52, cwd=True)

text(
    (60, 46.5),
    f"[{summary.severity}] {summary.title}",
    style=Styles.DarkBold.patch(text_size=11.5),
)

items = summary.action_items[:4]
rows = [["Priority", "Owner", "Remediation Task"]] + [
    [item.priority, item.owner, item.task] for item in items
]

table = Table(
    cell_style=Styles.White,
    text_style=Styles.Dark.patch(text_size=10.0),
    header_cell_style=Styles.PrimaryFlat,
    header_text_style=Styles.WhiteBold.patch(text_size=10.5),
    border_style=Styles.MutedThin,
)
table.set_style_cell_evenodd(
    even_color=Colors.Light,
    even_text_style=Styles.Dark.patch(text_size=10.0),
    odd_color=Colors.White,
    odd_text_style=Styles.Dark.patch(text_size=10.0),
)
table.set_style_cell_header(
    background_color=Colors.Primary,
    text_style=Styles.WhiteBold.patch(text_size=10.5),
)

row_heights = [9.0] + [27.0 / max(len(items), 1)] * len(items)
table.draw_flexible(
    xy=(6, 41),
    column_widths=[18, 26, 64],
    row_heights=row_heights,
    data=rows,
    align="top_left",
)

show()